# DNN (ANN) Model: Predicting EV Battery Failure Risk
Multi-class classification (Low / Medium / High) with a fully-connected neural network (Keras).

In [1]:
import os, random, json, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, roc_auc_score, balanced_accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED); random.seed(SEED); np.random.seed(SEED)
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers
FIG = os.environ.get("FIG_DIR", "figures"); os.makedirs(FIG, exist_ok=True)
CSV = os.environ.get("CSV_PATH", "ev_battery_health.csv")

## 1. Load data

In [2]:
df = pd.read_csv("ev_battery_health.csv", parse_dates=["manufacture_date", "observation_date", "last_service_date"])
print(df.shape)
print(df["failure_risk_label"].value_counts())
print(df["failure_risk_label"].value_counts(normalize=True).round(4))

(50000, 35)
failure_risk_label
Low       42735
Medium     4409
High       2856
Name: count, dtype: int64
failure_risk_label
Low       0.8547
Medium    0.0882
High      0.0571
Name: proportion, dtype: float64


## 2. Leakage check (important)
`capacity_fade_pct` equals `100 - state_of_health_pct` exactly, and the label is derived from SOH / RUL.
So SOH, RUL, capacity fade and degradation_category must NOT be inputs.

In [3]:
m = df.capacity_fade_pct.notna()
print("max |fade + SOH - 100| =", (df.loc[m, "capacity_fade_pct"] + df.loc[m, "state_of_health_pct"] - 100).abs().max())
print(df.groupby("failure_risk_label")[["state_of_health_pct", "remaining_useful_life_cycles"]].mean().round(1))

max |fade + SOH - 100| = 0.0
                    state_of_health_pct  remaining_useful_life_cycles
failure_risk_label                                                   
High                               73.1                           8.7
Low                                95.2                        1723.3
Medium                             83.5                         201.6


## 3. Feature engineering and selection

In [4]:
df["days_since_service"] = (df["observation_date"] - df["last_service_date"]).dt.days
df.loc[df["days_since_service"] < 0, "days_since_service"] = np.nan   # service date after observation = invalid
LEAKY = ["state_of_health_pct", "remaining_useful_life_cycles", "capacity_fade_pct", "degradation_category"]
IDS_DATES = ["battery_id", "observation_id", "manufacture_date", "observation_date", "last_service_date"]
CAT = ["chemistry", "manufacturer", "vehicle_model", "climate_zone", "bms_firmware_version"]
TARGET = "failure_risk_label"
NUM = [c for c in df.columns if c not in LEAKY + IDS_DATES + CAT + [TARGET]]
print(len(NUM), "numeric +", len(CAT), "categorical features")
print(NUM)

21 numeric + 5 categorical features
['pack_capacity_kwh', 'battery_age_days', 'cycle_count', 'avg_daily_distance_km', 'avg_charging_power_kw', 'fast_charge_ratio', 'avg_depth_of_discharge_pct', 'avg_state_of_charge_pct', 'charge_cycles_per_month', 'total_energy_throughput_kwh', 'avg_ambient_temp_c', 'humidity_pct', 'altitude_m', 'internal_resistance_mohm', 'voltage_avg_v', 'current_avg_a', 'charge_time_avg_min', 'peak_temp_during_fast_charge_c', 'service_count', 'recall_flag', 'days_since_service']


## 4. Train / validation / test split (grouped by battery)
One battery has ~19 observations; a random row split would put the same battery in train and test.

In [5]:
groups = df["battery_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, tmp_idx = next(gss.split(df, groups=groups))
tmp = df.iloc[tmp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_rel, te_rel = next(gss2.split(tmp, groups=tmp["battery_id"]))
train, val, test = df.iloc[tr_idx].copy(), tmp.iloc[va_rel].copy(), tmp.iloc[te_rel].copy()
assert not (set(train.battery_id) & set(test.battery_id)) and not (set(train.battery_id) & set(val.battery_id))
print(len(train), len(val), len(test))
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, d[TARGET].value_counts(normalize=True).round(3).to_dict())

34838 7604 7558
train {'Low': 0.855, 'Medium': 0.089, 'High': 0.056}
val {'Low': 0.852, 'Medium': 0.091, 'High': 0.057}
test {'Low': 0.856, 'Medium': 0.08, 'High': 0.063}


## 5. Preprocessing (fitted on training data only)
Winsorise at 1st/99th percentile -> log1p on skewed columns -> median impute (+ missing indicators) -> standardise;
categoricals: "Unknown" for missing, one-hot.

In [6]:
lo, hi = train[NUM].quantile(0.01), train[NUM].quantile(0.99)
LOG = ["cycle_count", "total_energy_throughput_kwh", "charge_time_avg_min", "internal_resistance_mohm",
       "battery_age_days", "altitude_m", "avg_daily_distance_km", "avg_charging_power_kw", "days_since_service", "service_count"]

def prep(d):
    d = d.copy()
    d[NUM] = d[NUM].clip(lower=lo, upper=hi, axis=1)
    d[LOG] = np.log1p(d[LOG])
    return d[NUM + CAT]

# outlier counts that winsorising touches (train)
print("values clipped (train):", int(((train[NUM] < lo) | (train[NUM] > hi)).sum().sum()))
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT)])
Xtr = pre.fit_transform(prep(train)).astype("float32")
Xva = pre.transform(prep(val)).astype("float32")
Xte = pre.transform(prep(test)).astype("float32")
le = LabelEncoder().fit(["Low", "Medium", "High"])      # High=0, Low=1, Medium=2 (alphabetical)
CLASSES = ["Low", "Medium", "High"]
cmap = {c: i for i, c in enumerate(CLASSES)}
ytr, yva, yte = [d[TARGET].map(cmap).values for d in (train, val, test)]
print("input dimension:", Xtr.shape[1])

values clipped (train): 11102
input dimension: 77


## 6. Model architecture

In [7]:
cw = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=ytr)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("class weights:", class_weight)

def build(input_dim):
    return keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(128, kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.30),
        layers.Dense(64, kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.30),
        layers.Dense(32, activation="relu"), layers.Dropout(0.20),
        layers.Dense(3, activation="softmax")], name="battery_risk_ann")
build(Xtr.shape[1]).summary()

class weights: {0: 0.3899092323361202, 1: 3.730377984794946, 2: 5.979745966357707}


Model: "battery_risk_ann"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         9,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,187 (82.76 KB)

 Trainable params: 20,803 (81.26 KB)

 Non-trainable params: 384 (1.50 KB)

## 7. Training
Class-weight strength (w = balanced_weight ** power) is the one hyper-parameter tuned, chosen by **validation** macro-F1.

In [8]:
def train_model(power):
    tf.keras.utils.set_random_seed(SEED)
    m = build(Xtr.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cw_p = {i: float(w) ** power for i, w in class_weight.items()}
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)]
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=150, batch_size=256, class_weight=cw_p, callbacks=cbs, verbose=0)
    return m, h, f1_score(yva, m.predict(Xva, verbose=0).argmax(1), average="macro")

tuning = {}
best = (None, None, -1, None)
for power in [0.0, 0.5, 1.0]:
    m_, h_, f1v = train_model(power); tuning[power] = round(f1v, 4)
    print(f"class-weight power={power}: val macro-F1={f1v:.4f}, epochs={len(h_.history['loss'])}")
    if f1v > best[2]: best = (m_, h_, f1v, power)
model, hist, _, BEST_POWER = best
print("selected power:", BEST_POWER)
print("epochs run:", len(hist.history["loss"]), "best val_loss:", min(hist.history["val_loss"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{FIG}/training_curves.png", dpi=150); plt.show()

class-weight power=0.0: val macro-F1=0.8528, epochs=32
class-weight power=0.5: val macro-F1=0.8354, epochs=33
class-weight power=1.0: val macro-F1=0.7992, epochs=33
selected power: 0.0
epochs run: 32 best val_loss: 0.14945244789123535


## 8. Evaluation on the held-out test set

In [9]:
p_te = model.predict(Xte, verbose=0); yp = p_te.argmax(1)
res = {"accuracy": accuracy_score(yte, yp), "balanced_accuracy": balanced_accuracy_score(yte, yp),
       "macro_f1": f1_score(yte, yp, average="macro"), "weighted_f1": f1_score(yte, yp, average="weighted"),
       "roc_auc_ovr_macro": roc_auc_score(yte, p_te, multi_class="ovr", average="macro")}
print({k: round(v, 4) for k, v in res.items()})
print(classification_report(yte, yp, target_names=CLASSES, digits=3))
cm = confusion_matrix(yte, yp); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0]); ax[0].set_title("Confusion matrix (counts)")
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[1]); ax[1].set_title("Confusion matrix (row-normalised)")
for a in ax: a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); plt.savefig(f"{FIG}/confusion_matrix.png", dpi=150); plt.show()

{'accuracy': 0.9512, 'balanced_accuracy': np.float64(0.8673), 'macro_f1': 0.8597, 'weighted_f1': 0.9518, 'roc_auc_ovr_macro': np.float64(0.986)}
              precision    recall  f1-score   support

         Low      0.982     0.976     0.979      6472
      Medium      0.688     0.728     0.708       607
        High      0.887     0.898     0.892       479

    accuracy                          0.951      7558
   macro avg      0.852     0.867     0.860      7558
weighted avg      0.953     0.951     0.952      7558



## 9. Baselines for context

In [10]:
rows = [{"model": "ANN (this work)", **{k: res[k] for k in ["accuracy", "balanced_accuracy", "macro_f1"]}}]
maj = np.full_like(yte, np.bincount(ytr).argmax())
rows.append({"model": "Majority class", "accuracy": accuracy_score(yte, maj), "balanced_accuracy": balanced_accuracy_score(yte, maj), "macro_f1": f1_score(yte, maj, average="macro")})
for name, clf in [("Logistic Regression", LogisticRegression(max_iter=2000, class_weight="balanced")),
                  ("Random Forest", RandomForestClassifier(300, class_weight="balanced", min_samples_leaf=3, n_jobs=-1, random_state=SEED))]:
    clf.fit(Xtr, ytr); q = clf.predict(Xte)
    rows.append({"model": name, "accuracy": accuracy_score(yte, q), "balanced_accuracy": balanced_accuracy_score(yte, q), "macro_f1": f1_score(yte, q, average="macro")})
comp = pd.DataFrame(rows).round(4); print(comp.to_string(index=False))

              model  accuracy  balanced_accuracy  macro_f1
    ANN (this work)    0.9512             0.8673    0.8597
     Majority class    0.8563             0.3333    0.3075
Logistic Regression    0.9202             0.8987    0.8290
      Random Forest    0.9476             0.8550    0.8509


## 10. Permutation feature importance (ANN, test set, macro-F1 drop)

In [11]:
names = pre.get_feature_names_out()
def macro(X): return f1_score(yte, model.predict(X, verbose=0).argmax(1), average="macro")
base = macro(Xte); rng = np.random.default_rng(SEED)
# group by original feature (numeric + its missing indicator; one-hot group per categorical)
groups_idx = {}
for i, n in enumerate(names):
    if n.startswith("num__"):
        groups_idx.setdefault(n.split("__")[1].replace("missingindicator_", ""), []).append(i)
    else:
        col = n.split("__")[1]
        for c in CAT:
            if col.startswith(c): groups_idx.setdefault(c, []).append(i)
imp = {}
for g, idx in groups_idx.items():
    drops = []
    for _ in range(3):
        Xp = Xte.copy(); perm = rng.permutation(len(Xp)); Xp[:, idx] = Xte[perm][:, idx]; drops.append(base - macro(Xp))
    imp[g] = np.mean(drops)
imp = pd.Series(imp).sort_values(ascending=False); print(imp.head(10).round(4))
top = imp.head(12)[::-1]
plt.figure(figsize=(7, 4.5)); plt.barh(top.index, top.values, color="#2a6f97"); plt.xlabel("Drop in test macro-F1 when shuffled")
plt.title("Top features (permutation importance)"); plt.tight_layout(); plt.savefig(f"{FIG}/feature_importance.png", dpi=150); plt.show()

internal_resistance_mohm       0.4148
cycle_count                    0.1791
chemistry                      0.1110
total_energy_throughput_kwh    0.0984
battery_age_days               0.0837
recall_flag                    0.0083
charge_cycles_per_month        0.0082
humidity_pct                   0.0078
pack_capacity_kwh              0.0076
avg_ambient_temp_c             0.0058
dtype: float64


## 11. EDA figure: class balance and key driver

In [12]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
df[TARGET].value_counts().reindex(CLASSES).plot.bar(ax=ax[0], color=["#52b788", "#f4a261", "#d62828"]); ax[0].set_title("Class distribution (full data)"); ax[0].set_xlabel("")
sns.boxplot(data=df, x=TARGET, y="cycle_count", order=CLASSES, ax=ax[1], showfliers=False, palette=["#52b788", "#f4a261", "#d62828"]); ax[1].set_title("Cycle count by failure risk")
plt.tight_layout(); plt.savefig(f"{FIG}/eda.png", dpi=150); plt.show()

json.dump({"results": res, "comparison": comp.to_dict("records"), "epochs": len(hist.history["loss"]),
           "n_params": int(model.count_params()), "input_dim": int(Xtr.shape[1]),
           "sizes": [len(train), len(val), len(test)], "class_weight": class_weight, "tuning": tuning, "best_power": BEST_POWER,
           "per_class": classification_report(yte, yp, target_names=CLASSES, output_dict=True),
           "cm": cm.tolist(), "top_features": imp.head(8).round(4).to_dict(),
           "best_val_loss": float(min(hist.history["val_loss"])), "final_train_acc": float(hist.history["accuracy"][-1]),
           "final_val_acc": float(hist.history["val_accuracy"][-1])}, open("results.json", "w"), indent=1, default=float)

In [14]:

import joblib

bundle = {
    # ANN stored as architecture + weights (safely picklable)
    "keras_architecture": model.to_json(),
    "keras_weights": model.get_weights(),
    # preprocessing fitted on the training set
    "preprocessor": pre,
    "clip_lower": lo,
    "clip_upper": hi,
    "num_cols": NUM,
    "cat_cols": CAT,
    "log_cols": LOG,
    "classes": CLASSES,   # index 0=Low, 1=Medium, 2=High
    "metadata": {
        "task": "EV battery failure-risk classification",
        "test_accuracy": float(res["accuracy"]),
        "test_macro_f1": float(res["macro_f1"]),
        "tensorflow": tf.__version__,
        "random_seed": SEED,
    },
}
joblib.dump(bundle, "ev_battery_risk_model.joblib", compress=3)
print("Saved ev_battery_risk_model.joblib")

# Sanity check: reload the bundle and confirm identical predictions
b = joblib.load("ev_battery_risk_model.joblib")
m2 = keras.models.model_from_json(b["keras_architecture"])
m2.set_weights(b["keras_weights"])
assert np.allclose(model.predict(Xte[:200], verbose=0), m2.predict(Xte[:200], verbose=0), atol=1e-5)
print("Reload check passed")

Saved ev_battery_risk_model.joblib
Reload check passed
